# 02 — Telugu ASR Model Pipeline Prototyping
## Wav2Vec2 XLS-R 300M · CTC · Prototype Validation
**Prerequisites (from File 1):**
- `./telugu_asr_clean_dataset/` — cleaned HuggingFace dataset (columns: `audio`, `transcription`, `audio_id`)
- `./character_inventory.json` — character inventory with 64 unique Telugu characters

**Goal:** Validate the full ASR pipeline end-to-end on a 500-sample subset before full training.

## Section 1 — Install Dependencies

In [1]:
# Pin datasets<3.0 — datasets>=3.0 uses torchcodec as its audio backend, which requires
# FFmpeg shared libraries (libavutil.so.*) that are not present on this machine.
# datasets<3.0 uses soundfile for audio I/O, which works with the installed environment.
%pip install -q --force-reinstall "datasets<3.0"
%pip install -q transformers torch evaluate jiwer accelerate soundfile librosa
# torchaudio excluded — compiled against CUDA 13 (libcudart.so.13) which is not present here

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Section 2 — GPU Check

In [1]:
import torch

# Verify CUDA is available (RTX A6000 — 49 GB VRAM)
assert torch.cuda.is_available(), "No CUDA device found — check driver/runtime."
DEVICE = torch.device("cuda")

print(f"PyTorch version   : {torch.__version__}")
print(f"CUDA version      : {torch.version.cuda}")
print(f"Device name       : {torch.cuda.get_device_name(0)}")
print(f"Total VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Device            : {DEVICE}")

PyTorch version   : 2.6.0+cu124
CUDA version      : 12.4
Device name       : NVIDIA RTX A6000
Total VRAM        : 51.0 GB
Device            : cuda


## Section 3 — Load Cleaned Dataset & Extract 500-Record Subset

In [2]:
from datasets import load_from_disk, Audio

# Paths produced by File 1
CLEAN_DATASET_PATH: str = "./telugu_asr_clean_dataset/"
CHARACTER_INVENTORY_PATH: str = "./character_inventory.json"
PROTO_SUBSET_SIZE: int = 500

# Load full clean dataset (35,426 samples)
full_dataset = load_from_disk(CLEAN_DATASET_PATH)

# Re-cast audio column to 16 kHz — datasets uses soundfile for decoding (torchaudio not required)
# Note: mono=True was added in a later datasets version; omit for compatibility
full_dataset = full_dataset.cast_column("audio", Audio(sampling_rate=16_000))

print(f"Full dataset      : {len(full_dataset)} samples")
print(f"Columns           : {full_dataset.column_names}")
print(f"Features          : {full_dataset.features}")

# Deterministic 500-sample prototype subset
proto_dataset = full_dataset.select(range(PROTO_SUBSET_SIZE))
print(f"\nPrototype subset  : {len(proto_dataset)} samples")

# Column-wise access — avoids triggering audio decode at load time
print(f"Sample transcription: {full_dataset['transcription'][0]}")
print(f"Sample audio_id     : {full_dataset['audio_id'][0]}")

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Full dataset      : 35426 samples
Columns           : ['audio', 'transcription', 'audio_id']
Features          : {'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None), 'transcription': Value(dtype='string', id=None), 'audio_id': Value(dtype='string', id=None)}

Prototype subset  : 500 samples
Sample transcription: కచ్చితంగా చూపిస్తుంది కదా మరి
Sample audio_id     : TE2406-TE2408_1-A.089.wav


## Section 4 — Build Vocabulary from Full Dataset

In [3]:
import json

VOCAB_SAVE_PATH: str = "./vocab.json"

# --- Collect all unique characters from the FULL dataset ---
# Column-wise access returns a plain list of strings WITHOUT decoding any audio.
# The old row-wise loop (`for sample in full_dataset`) decoded all 35k audio files
# unnecessarily and was the main source of codec errors.
all_chars: set = set()
for text in full_dataset["transcription"]:
    # Space → "|" word-boundary token; exclude raw spaces from vocab
    all_chars.update(set(text.replace(" ", "")))

# Sort for determinism
sorted_chars: list = sorted(all_chars)

# Build char→id mapping starting at 0
vocab: dict = {char: idx for idx, char in enumerate(sorted_chars)}

# Add special tokens after regular characters
# "|" = word boundary (replaces space during tokenisation)
# "[UNK]" = unknown token
# "[PAD]" = CTC blank / padding token
vocab["|"]     = len(vocab)
vocab["[UNK]"] = len(vocab)
vocab["[PAD]"] = len(vocab)

# Save vocab.json with UTF-8 to preserve Telugu codepoints
with open(VOCAB_SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

print(f"Vocabulary size   : {len(vocab)} tokens")
print(f"  Regular chars   : {len(sorted_chars)}")
print(f"  Special tokens  : | [UNK] [PAD]")
print(f"  Saved to        : {VOCAB_SAVE_PATH}")
print(f"\nFirst 10 entries  : { {k: v for k, v in list(vocab.items())[:10]} }")
print(f"Special tokens    : '|'→{vocab['|']}, '[UNK]'→{vocab['[UNK]']}, '[PAD]'→{vocab['[PAD]']}")

Vocabulary size   : 67 tokens
  Regular chars   : 64
  Special tokens  : | [UNK] [PAD]
  Saved to        : ./vocab.json

First 10 entries  : {'ం': 0, 'ః': 1, 'అ': 2, 'ఆ': 3, 'ఇ': 4, 'ఈ': 5, 'ఉ': 6, 'ఊ': 7, 'ఋ': 8, 'ఎ': 9}
Special tokens    : '|'→64, '[UNK]'→65, '[PAD]'→66


## Section 5 — Initialize Processor (Tokenizer + Feature Extractor)

In [4]:
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)

PROCESSOR_SAVE_PATH: str = "./telugu_wav2vec2_processor/"

# -- Tokenizer: maps Telugu characters ↔ integer IDs --
tokenizer = Wav2Vec2CTCTokenizer(
    VOCAB_SAVE_PATH,
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

# -- Feature Extractor: raw waveform → normalised float array --
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16_000,
    padding_value=0.0,
    do_normalize=True,          # zero-mean / unit-variance per utterance
    return_attention_mask=True,
)

# -- Processor: thin wrapper combining both --
processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)
processor.save_pretrained(PROCESSOR_SAVE_PATH)
print(f"Processor saved to: {PROCESSOR_SAVE_PATH}")
print(f"Vocab size        : {processor.tokenizer.vocab_size}")

# -- Sanity test: encode then decode a Telugu sentence --
TEST_SENTENCE: str = "కచ్చితంగా చూపిస్తుంది"
token_ids = processor.tokenizer(TEST_SENTENCE).input_ids
decoded   = processor.tokenizer.decode(token_ids)
print(f"\nSanity test")
print(f"  Input  : {TEST_SENTENCE}")
print(f"  IDs    : {token_ids}")
print(f"  Decoded: {decoded}")
assert decoded == TEST_SENTENCE, f"Round-trip mismatch: '{decoded}' != '{TEST_SENTENCE}'"
print("  Round-trip: PASS")

Processor saved to: ./telugu_wav2vec2_processor/
Vocab size        : 67

Sanity test
  Input  : కచ్చితంగా చూపిస్తుంది
  IDs    : [15, 20, 61, 20, 50, 30, 0, 17, 49, 64, 20, 53, 35, 50, 47, 61, 30, 52, 0, 32, 50]
  Decoded: కచ్చితంగా చూపిస్తుంది
  Round-trip: PASS


## Section 6 — Prepare Dataset (Feature Extraction + Label Encoding)

In [5]:
def prepare_dataset(batch: dict) -> dict:
    """
    Convert a raw dataset batch into model-ready tensors.

    Input  columns: audio (dict with 'array' & 'sampling_rate'), transcription, audio_id
    Output columns: input_values (float32 array), labels (int list), input_length (int)
    """
    audio = batch["audio"]

    # Feature extraction: waveform → normalised float array (no padding here; done at collation)
    batch["input_values"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_values[0]
    batch["input_length"] = len(batch["input_values"])

    # Tokenise transcription: str → list[int]
    # IMPORTANT: spaces in text map to "|" (word-boundary token), not token-id 0
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids

    return batch

# Map over the 500-record prototype subset; drop original columns we no longer need
COLUMNS_TO_REMOVE: list = ["audio", "transcription", "audio_id"]

prepared_dataset = proto_dataset.map(
    prepare_dataset,
    remove_columns=COLUMNS_TO_REMOVE,
    desc="Preparing features",
)

print(f"Prepared dataset columns : {prepared_dataset.column_names}")
print(f"Sample input_values shape: ({prepared_dataset[0]['input_length']},)")
print(f"Sample labels            : {prepared_dataset[0]['labels'][:10]} ...")

Preparing features: 100%|██████████| 500/500 [00:14<00:00, 35.64 examples/s]

Prepared dataset columns : ['input_values', 'input_length', 'labels']
Sample input_values shape: (30592,)
Sample labels            : [15, 20, 61, 20, 50, 30, 0, 17, 49, 64] ...


## Section 7 — Load Wav2Vec2 XLS-R 300M Model

In [7]:
from huggingface_hub import login

# Paste your HF token here (Settings → Access Tokens on huggingface.co)
# Enables authenticated downloads: higher rate limits, private repos, faster CDN
HF_TOKEN: str = os.environ.get("HF_TOKEN", "")  # Set via: export HF_TOKEN=hf_...

login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ Logged in to HuggingFace Hub.")

✅ Logged in to HuggingFace Hub.


In [8]:
from transformers import Wav2Vec2ForCTC

PRETRAINED_MODEL_ID: str = "facebook/wav2vec2-xls-r-300m"
VOCAB_SIZE: int = processor.tokenizer.vocab_size

model = Wav2Vec2ForCTC.from_pretrained(
    PRETRAINED_MODEL_ID,
    # --- Regularisation ---
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    # --- SpecAugment (applied on hidden states during training) ---
    mask_time_prob=0.05,        # probability of masking a time step
    mask_time_length=10,        # consecutive frames masked per span
    # --- CTC head ---
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,     # replace inf CTC losses with 0 (avoids NaN)
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=VOCAB_SIZE,
    ignore_mismatched_sizes=True,  # resize lm_head to our vocab
)

# Freeze CNN feature encoder — its weights are already well-trained for audio
# Only the Transformer stack and CTC head will be fine-tuned
model.freeze_feature_encoder()
model = model.to(DEVICE)

# Parameter count
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model             : {PRETRAINED_MODEL_ID}")
print(f"Total params      : {total_params:,}")
print(f"Trainable params  : {trainable_params:,}  ({100*trainable_params/total_params:.1f}%)")
print(f"Vocab size        : {VOCAB_SIZE}")
print(f"Device            : {next(model.parameters()).device}")

Loading weights: 100%|██████████| 422/422 [00:00<00:00, 9732.37it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model             : facebook/wav2vec2-xls-r-300m
Total params      : 315,507,395
Trainable params  : 311,297,219  (98.7%)
Vocab size        : 67
Device            : cuda:0


## Section 8 — Dummy Forward Pass (Shape & Loss Verification)

In [9]:
import torch
from torch.nn.utils.rnn import pad_sequence

# --- Grab 2 samples from prepared dataset ---
samples = [prepared_dataset[i] for i in range(2)]

# Pad input_values to the same length (right-pad with 0.0)
input_tensors = [torch.tensor(s["input_values"], dtype=torch.float32) for s in samples]
input_values  = pad_sequence(input_tensors, batch_first=True, padding_value=0.0).to(DEVICE)

# Pad labels with -100 (NOT 0 — token-id 0 is a real Telugu character; -100 is ignored by CTC loss)
label_tensors = [torch.tensor(s["labels"], dtype=torch.long) for s in samples]
labels        = pad_sequence(label_tensors, batch_first=True, padding_value=-100).to(DEVICE)

# Attention mask: 1 for real tokens, 0 for padding
attention_mask = (input_values != 0.0).long()

print(f"input_values shape : {tuple(input_values.shape)}  (batch, time_steps)")
print(f"labels shape       : {tuple(labels.shape)}        (batch, label_len)")
print(f"attention_mask sum : {attention_mask.sum(dim=1).tolist()}")

# Forward pass
model.eval()
with torch.no_grad():
    outputs = model(
        input_values=input_values,
        attention_mask=attention_mask,
        labels=labels,
    )

logits = outputs.logits
loss   = outputs.loss

print(f"\nlogits shape       : {tuple(logits.shape)}  (batch, frames, vocab_size)")
print(f"loss               : {loss.item():.4f}")

# Critical check: last dim must equal vocabulary size
assert logits.shape[-1] == VOCAB_SIZE, \
    f"logits last dim {logits.shape[-1]} != vocab_size {VOCAB_SIZE}"
print(f"Assertion passed   : logits last dim == vocab_size ({VOCAB_SIZE})")

input_values shape : (2, 39984)  (batch, time_steps)
labels shape       : (2, 29)        (batch, label_len)
attention_mask sum : [30592, 39984]

logits shape       : (2, 124, 67)  (batch, frames, vocab_size)
loss               : 18.4530
Assertion passed   : logits last dim == vocab_size (67)


## Section 9 — Dummy Training Loop (3–5 Steps)

In [10]:
from torch.optim import AdamW

NUM_DUMMY_STEPS: int = 5
LEARNING_RATE: float = 3e-5
DUMMY_BATCH_SIZE: int = 2  # re-use the 2-sample batch from Section 8

# AdamW — standard optimiser for Transformer fine-tuning
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE,
)

model.train()
loss_history: list = []

for step in range(1, NUM_DUMMY_STEPS + 1):
    optimizer.zero_grad()

    # Rebuild batch each step (same 2 samples — this is a smoke-test, not real training)
    input_tensors = [torch.tensor(prepared_dataset[i]["input_values"], dtype=torch.float32)
                     for i in range(DUMMY_BATCH_SIZE)]
    label_tensors = [torch.tensor(prepared_dataset[i]["labels"], dtype=torch.long)
                     for i in range(DUMMY_BATCH_SIZE)]

    input_values_train = pad_sequence(input_tensors, batch_first=True, padding_value=0.0).to(DEVICE)
    labels_train       = pad_sequence(label_tensors, batch_first=True, padding_value=-100).to(DEVICE)
    attn_mask_train    = (input_values_train != 0.0).long()

    # Forward
    out = model(
        input_values=input_values_train,
        attention_mask=attn_mask_train,
        labels=labels_train,
    )
    step_loss = out.loss

    # Backward + update
    step_loss.backward()
    optimizer.step()

    loss_history.append(step_loss.item())
    print(f"  Step {step}/{NUM_DUMMY_STEPS}  |  loss = {step_loss.item():.4f}")

print(f"\nLoss trajectory: {[f'{l:.4f}' for l in loss_history]}")
print("Loss is decreasing:", loss_history[-1] < loss_history[0])

# --- Decode last step's logits to verify the full decode pathway ---
model.eval()
with torch.no_grad():
    final_out    = model(input_values=input_values_train, attention_mask=attn_mask_train)
    final_logits = final_out.logits                          # (batch, frames, vocab)

# Greedy argmax decode
pred_ids      = torch.argmax(final_logits, dim=-1)          # (batch, frames)
decoded_preds = processor.batch_decode(pred_ids)

print(f"\nDecoded predictions from last step:")
for i, pred in enumerate(decoded_preds):
    print(f"  Sample {i}: '{pred}'")

  Step 1/5  |  loss = 18.4867
  Step 2/5  |  loss = 18.4422
  Step 3/5  |  loss = 18.2200
  Step 4/5  |  loss = 17.9423
  Step 5/5  |  loss = 18.1631

Loss trajectory: ['18.4867', '18.4422', '18.2200', '17.9423', '18.1631']
Loss is decreasing: True

Decoded predictions from last step:
  Sample 0: 'ైైైైైైైైైైైైైఙ'
  Sample 1: 'ైైైైైైైైైై'


## Section 10 — Pipeline Summary

In [11]:
print("=" * 60)
print("  PIPELINE VALIDATION SUMMARY")
print("=" * 60)

print(f"\n[Vocabulary]")
print(f"  Vocab size                : {VOCAB_SIZE}")
print(f"  Special tokens            : '|' (word boundary), '[UNK]', '[PAD]'")
print(f"  PAD token id              : {processor.tokenizer.pad_token_id}")

print(f"\n[Model — {PRETRAINED_MODEL_ID}]")
print(f"  Total parameters          : {total_params:,}")
print(f"  Trainable parameters      : {trainable_params:,}  ({100*trainable_params/total_params:.1f}%)")
print(f"  Feature encoder           : FROZEN")

print(f"\n[Tensor Shapes — Forward Pass]")
print(f"  input_values              : {tuple(input_values.shape)}")
print(f"  labels (padded w/ -100)   : {tuple(labels.shape)}")
print(f"  logits                    : {tuple(logits.shape)}")
print(f"  logits last dim == vocab  : {logits.shape[-1] == VOCAB_SIZE}")

print(f"\n[Dummy Training Loop — {NUM_DUMMY_STEPS} steps, lr={LEARNING_RATE}]")
for i, l in enumerate(loss_history, 1):
    print(f"  Step {i}: {l:.4f}")
print(f"  Loss decreased            : {loss_history[-1] < loss_history[0]}")

print(f"\n[Status]")
print(f"  Round-trip encode/decode  : PASS")
print(f"  Forward pass              : PASS")
print(f"  Backward pass / grad step : PASS")
print(f"  CTC decode                : PASS")
print(f"\n  Pipeline is ready for full fine-tuning (File 3).")
print("=" * 60)

  PIPELINE VALIDATION SUMMARY

[Vocabulary]
  Vocab size                : 67
  Special tokens            : '|' (word boundary), '[UNK]', '[PAD]'
  PAD token id              : 66

[Model — facebook/wav2vec2-xls-r-300m]
  Total parameters          : 315,507,395
  Trainable parameters      : 311,297,219  (98.7%)
  Feature encoder           : FROZEN

[Tensor Shapes — Forward Pass]
  input_values              : (2, 39984)
  labels (padded w/ -100)   : (2, 29)
  logits                    : (2, 124, 67)
  logits last dim == vocab  : True

[Dummy Training Loop — 5 steps, lr=3e-05]
  Step 1: 18.4867
  Step 2: 18.4422
  Step 3: 18.2200
  Step 4: 17.9423
  Step 5: 18.1631
  Loss decreased            : True

[Status]
  Round-trip encode/decode  : PASS
  Forward pass              : PASS
  Backward pass / grad step : PASS
  CTC decode                : PASS

  Pipeline is ready for full fine-tuning (File 3).
